## Step 1 — Install everything

Every package this notebook needs gets installed here, in one block, with **no imports
in between**. That's what lets the numpy/pandas ABI fix at the end of this cell take
effect without a kernel restart: as long as numpy/pandas/nibabel have never been imported
yet in this kernel's lifetime, the first import later on will simply pick up whatever
final, consistent versions are on disk after this cell finishes. The restart was only ever
needed because installs and imports used to be interleaved.

In [1]:
# Kaggle already ships torch, numpy, scipy, huggingface_hub
!pip install -q brainles-preprocessing
!pip install -q monai safetensors nibabel neuroHarmonize
!pip install -q -U huggingface_hub

# Run this LAST, after the packages above, since some of their dependency resolution
# can silently drift numpy/pandas to mismatched versions. This re-pins them to a
# mutually consistent pair as the final word on what's installed.
!pip install -q --upgrade --force-reinstall --no-cache-dir numpy pandas

print("All installs complete.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.0/244.0 kB 19.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-clo

## Step 2 — First imports of the session (numpy/pandas/nibabel etc. have not been touched until now)

In [2]:
import glob
import os
import tarfile

import numpy as np
import pandas as pd
import nibabel as nib
from tqdm import tqdm

print("numpy:", np.__version__, "| pandas:", pd.__version__)

numpy: 2.0.2 | pandas: 3.0.5


## Hugging Face auth for BrainIAC

In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF-TOKEN")
login(token=hf_token)

In [4]:
from huggingface_hub import hf_hub_download

backbone_path = hf_hub_download(repo_id="eugenehp/brainiac", filename="backbone.safetensors")
idh_head_path = hf_hub_download(repo_id="eugenehp/brainiac", filename="idh.safetensors")

backbone.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

backbone.safetensors: downloading bytes:           |  0.00B            

idh.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

idh.safetensors: downloading bytes:           |  0.00B            

## Dataset paths

In [5]:
brats_root = "/kaggle/input/datasets/dschettler8845/brats-2021-task1"
ucsf_root  = "/kaggle/input/datasets/usmansadiqcs/ucsf-pdgm-v3-dataset"

brats_extract_dir = "/kaggle/working/brats2021"
ucsf_extract_dir  = "/kaggle/working/ucsf_pdgm"

print("BraTS root contents:", os.listdir(brats_root)[:5])
print("UCSF root contents:", os.listdir(ucsf_root)[:5])

BraTS root contents: ['BraTS2021_00495.tar', 'BraTS2021_Training_Data.tar', 'BraTS2021_00621.tar']
UCSF root contents: ['UCSF-PDGM-metadata.csv', 'UCSF-PDGM']


## Extract BraTS2021 (packaged as `.tar` archives, ~13.4 GB uncompressed — fits Kaggle scratch disk)

In [6]:
os.makedirs(brats_extract_dir, exist_ok=True)

tar_files = glob.glob(f"{brats_root}/*.tar")
print(f"Found {len(tar_files)} tar files:", [os.path.basename(t) for t in tar_files])

for tar_path in tar_files:
    print(f"Extracting {os.path.basename(tar_path)} ...")
    with tarfile.open(tar_path, "r") as tf:
        for member in tqdm(tf.getmembers(), desc=os.path.basename(tar_path)):
            tf.extract(member, brats_extract_dir)

n_extracted = sum(len(files) for _, _, files in os.walk(brats_extract_dir))
print("Done. Total files extracted:", n_extracted)

Found 3 tar files: ['BraTS2021_00495.tar', 'BraTS2021_Training_Data.tar', 'BraTS2021_00621.tar']
Extracting BraTS2021_00495.tar ...


BraTS2021_00495.tar:   0%|          | 0/6 [00:00<?, ?it/s]/tmp/ipykernel_24/607796597.py:10: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extract(member, brats_extract_dir)
BraTS2021_00495.tar: 100%|██████████| 6/6 [00:00<00:00, 28.49it/s]


Extracting BraTS2021_Training_Data.tar ...


BraTS2021_Training_Data.tar: 100%|██████████| 7508/7508 [00:57<00:00, 130.73it/s]


Extracting BraTS2021_00621.tar ...


BraTS2021_00621.tar: 100%|██████████| 6/6 [00:00<00:00, 120.83it/s]


Done. Total files extracted: 6266


## Locate UCSF-PDGM files

Checks for several possible packaging formats (loose `.nii`/`.nii.gz`, `.tar`, `.tar.gz`/`.tgz`,
`.zip`) since different Kaggle re-uploads of the same dataset package it differently. If none
of those match, it prints a full directory tree and file-extension breakdown instead of just
failing — so if this still can't find anything, we'll know exactly what's actually there from
the printed output rather than guessing again.

In [7]:
import zipfile
from collections import Counter


def describe_dir(path, max_depth=3, max_entries=8):
    """Print a top-level listing, a shallow tree, and file-extension counts for diagnosis."""
    print(f"--- Top-level listing of {path} ---")
    try:
        print(os.listdir(path))
    except Exception as e:
        print(f"Could not list {path}: {e}")
        return

    print(f"\n--- Tree (depth <= {max_depth}) ---")
    for root, dirs, files in os.walk(path):
        depth = root[len(path):].count(os.sep)
        if depth > max_depth:
            dirs[:] = []
            continue
        indent = "  " * depth
        print(f"{indent}{os.path.basename(root) or root}/")
        for f in files[:max_entries]:
            print(f"{indent}  {f}")
        if len(files) > max_entries:
            print(f"{indent}  ... +{len(files) - max_entries} more")

    print("\n--- File extension counts (all files, any depth) ---")
    all_files = []
    for root, dirs, files in os.walk(path):
        all_files.extend(files)
    ext_counts = Counter(
        f[f.index('.'):] if '.' in f else '(no extension)'
        for f in all_files
    )
    for ext, count in ext_counts.most_common(15):
        print(f"  {ext}: {count}")


nii_gz     = [p for p in glob.glob(f"{ucsf_root}/**/*.nii.gz", recursive=True) if os.path.isfile(p)]
nii_plain  = [p for p in glob.glob(f"{ucsf_root}/**/*.nii", recursive=True) if os.path.isfile(p)]
tar_files  = (glob.glob(f"{ucsf_root}/**/*.tar", recursive=True)
              + glob.glob(f"{ucsf_root}/**/*.tar.gz", recursive=True)
              + glob.glob(f"{ucsf_root}/**/*.tgz", recursive=True))
zip_files  = glob.glob(f"{ucsf_root}/**/*.zip", recursive=True)

if nii_gz or nii_plain:
    n = len(nii_gz) + len(nii_plain)
    print(f"Found {n} NIfTI files somewhere under ucsf_root (possibly nested inside "
          f"per-modality wrapper folders — see the next section). No archive extraction needed.")
    ucsf_data_dir = ucsf_root

elif tar_files:
    print(f"Found {len(tar_files)} tar-family archives, extracting ...")
    os.makedirs(ucsf_extract_dir, exist_ok=True)
    for tar_path in tar_files:
        print(f"Extracting {os.path.basename(tar_path)} ...")
        with tarfile.open(tar_path, "r:*") as tf:  # r:* auto-detects gz/bz2/plain
            for member in tqdm(tf.getmembers(), desc=os.path.basename(tar_path)):
                tf.extract(member, ucsf_extract_dir)
    ucsf_data_dir = ucsf_extract_dir

elif zip_files:
    print(f"Found {len(zip_files)} zip archives, extracting ...")
    os.makedirs(ucsf_extract_dir, exist_ok=True)
    for zip_path in zip_files:
        print(f"Extracting {os.path.basename(zip_path)} ...")
        with zipfile.ZipFile(zip_path, "r") as zf:
            for member in tqdm(zf.namelist(), desc=os.path.basename(zip_path)):
                zf.extract(member, ucsf_extract_dir)
    ucsf_data_dir = ucsf_extract_dir

else:
    describe_dir(ucsf_root)
    raise FileNotFoundError(
        "No .nii/.nii.gz/.tar/.tar.gz/.tgz/.zip files found anywhere under ucsf_root. "
        "See the directory tree and extension counts printed above to figure out the "
        "actual packaging format, then adjust the glob patterns above accordingly."
    )

print("UCSF data directory set to:", ucsf_data_dir)

Found 1503 NIfTI files somewhere under ucsf_root (possibly nested inside per-modality wrapper folders — see the next section). No archive extraction needed.
UCSF data directory set to: /kaggle/input/datasets/usmansadiqcs/ucsf-pdgm-v3-dataset


## Sanity checks — load one sample volume from each dataset, plus the clinical CSV

In [8]:
# BraTS sample
sample_t1ce = glob.glob(f"{brats_extract_dir}/**/*_t1ce.nii.gz", recursive=True)[0]
img = nib.load(sample_t1ce)
print("BraTS T1CE shape/spacing:", img.shape, img.header.get_zooms())

BraTS T1CE shape/spacing: (240, 240, 155) (np.float32(1.0), np.float32(1.0), np.float32(1.0))


### UCSF-PDGM modality lookup

This Kaggle mirror packages each modality as a **directory** ending in `.nii`
(e.g. `UCSF-PDGM-0233_T1c_bias.nii/`) with the real NIfTI file one level inside it —
and that inner filename isn't always consistent (`T1gad` is used as an alias for `T1c`
in places, and at least one follow-up case folder contains a file whose embedded case ID
doesn't match the folder's case ID). `find_modality_file` below matches on the *parent
folder name* rather than the leaf filename, and prints a warning if the case ID inside
the file doesn't match the folder it's sitting in — worth keeping that check anywhere
else in the pipeline that reads this dataset, not just here.

In [9]:
def find_modality_file(case_dir, modality_keywords, verify_case_id=True):
    """
    Return the path to a modality's NIfTI file within one UCSF-PDGM case folder.

    Matches on the PARENT directory name (case-insensitive substring match against
    modality_keywords), then returns whichever single NIfTI file sits inside it —
    since the leaf filename itself can use an inconsistent alias (e.g. 'T1gad' for 'T1c').
    """
    case_id = os.path.basename(case_dir).split("_nifti")[0]
    for root, dirs, files in os.walk(case_dir):
        parent_name = os.path.basename(root).lower()
        if any(kw.lower() in parent_name for kw in modality_keywords):
            nii_files = [f for f in files if f.endswith((".nii", ".nii.gz"))]
            if nii_files:
                match_path = os.path.join(root, nii_files[0])
                if verify_case_id and case_id not in nii_files[0]:
                    print(f"  WARNING: folder is for case {case_id} but the file inside "
                          f"is {nii_files[0]!r} — case ID mismatch, verify before using.")
                return match_path
    return None


case_dirs = sorted(glob.glob(f"{ucsf_data_dir}/**/UCSF-PDGM-*_nifti", recursive=True))
if not case_dirs:
    describe_dir(ucsf_data_dir)
    raise FileNotFoundError("No folders matching 'UCSF-PDGM-*_nifti' found under ucsf_data_dir.")

print(f"Found {len(case_dirs)} case folders. Using the first as a sample: {os.path.basename(case_dirs[0])}")

sample_t1c = find_modality_file(case_dirs[0], ["t1c", "t1gad"])
if sample_t1c is None:
    raise FileNotFoundError(f"No T1c/T1gad modality folder found inside {case_dirs[0]}")

img2 = nib.load(sample_t1c)
print("UCSF T1c shape/spacing:", img2.shape, img2.header.get_zooms())
print("Loaded from:", sample_t1c)

Found 501 case folders. Using the first as a sample: UCSF-PDGM-0004_nifti
UCSF T1c shape/spacing: (240, 240, 155) (np.float32(1.0), np.float32(1.0), np.float32(1.0))
Loaded from: /kaggle/input/datasets/usmansadiqcs/ucsf-pdgm-v3-dataset/UCSF-PDGM/UCSF-PDGM-0004_nifti/UCSF-PDGM-0004_T1c_bias.nii/UCSF-PDGM-0004_T1gad_bias.nii


In [10]:
# UCSF clinical metadata
clinical_csv_candidates = glob.glob(f"{ucsf_data_dir}/**/*metadata*.csv", recursive=True)
if not clinical_csv_candidates:
    # fall back to any csv at all, in case this re-upload names it differently
    clinical_csv_candidates = glob.glob(f"{ucsf_data_dir}/**/*.csv", recursive=True)

if not clinical_csv_candidates:
    describe_dir(ucsf_data_dir)
    raise FileNotFoundError("No CSV files found under ucsf_data_dir — see the tree printed above.")

clinical_csv = clinical_csv_candidates[0]
print("Using clinical CSV:", clinical_csv)
df = pd.read_csv(clinical_csv)

print(df[["ID", "Sex", "Age at MRI", "IDH", "MGMT status"]].head())
print(df["IDH"].value_counts())

Using clinical CSV: /kaggle/input/datasets/usmansadiqcs/ucsf-pdgm-v3-dataset/UCSF-PDGM-metadata.csv
              ID Sex  Age at MRI       IDH    MGMT status
0  UCSF-PDGM-004   M          66  wildtype       negative
1  UCSF-PDGM-005   F          80  wildtype  indeterminate
2  UCSF-PDGM-007   M          70  wildtype  indeterminate
3  UCSF-PDGM-008   M          70  wildtype       negative
4  UCSF-PDGM-009   F          68  wildtype       negative
IDH
wildtype            398
IDH1 p.R132H         54
mutated (NOS)        31
IDH1 p.R132C          8
IDH1 p.R132G          5
IDH1 p.R132S          2
IDH2 p.R172K          1
IDH2 p.Arg172Trp      1
IDH1 p.Arg132His      1
Name: count, dtype: int64


### Case-ID mismatch scan — all case folders

The check above only covers the sample case, `UCSF-PDGM-0004`. Since we already found one
folder/file case-ID mismatch just by eyeballing the tree output (`0433_FU007d` → `0315`),
this cell runs `find_modality_file` — same `t1c`/`t1gad` keywords — across every folder in
`case_dirs` and sorts the result into two lists:

- `mismatch_log` — folder was found, but the case ID embedded in the file inside doesn't match it
- `missing_log` — no T1c/T1gad modality folder found in that case at all

No NIfTI data gets loaded here, just filesystem walks, so it's cheap to run across all folders.
If `mismatch_log` comes back as basically just `0433_FU007d`, that's a one-off to exclude or fix;
if it's a real chunk of the 501, treat it as a data-provenance issue worth tracking down rather
than something to quietly drop.

In [11]:
import contextlib
import io

modality_keywords = ["t1c", "t1gad"]  # same keywords used in the sample check above

mismatch_log = []
missing_log = []

for cdir in tqdm(case_dirs, desc="Scanning case folders"):
    case_id = os.path.basename(cdir).split("_nifti")[0]
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        matched_path = find_modality_file(cdir, modality_keywords)
    warning_text = buf.getvalue().strip()

    if matched_path is None:
        missing_log.append({"case_id": case_id, "case_dir": cdir})
    elif warning_text:
        mismatch_log.append({
            "case_id": case_id,
            "case_dir": cdir,
            "matched_file": os.path.basename(matched_path),
            "warning": warning_text,
        })

print(f"Scanned {len(case_dirs)} case folders.")
print(f"Case-ID mismatches: {len(mismatch_log)}")
print(f"No T1c/T1gad folder found at all: {len(missing_log)}")

mismatch_df = pd.DataFrame(mismatch_log, columns=["case_id", "case_dir", "matched_file", "warning"])
if len(mismatch_df):
    print()
    print(mismatch_df)
else:
    print("\nNo case-ID mismatches found beyond the one already spotted — looks like it was isolated.")

if missing_log:
    print("\nFolders with no T1c/T1gad modality found at all:")
    for m in missing_log:
        print(" ", m["case_id"])

mismatch_df.to_csv("/kaggle/working/ucsf_case_id_mismatches.csv", index=False)
print("\nSaved mismatch log to /kaggle/working/ucsf_case_id_mismatches.csv")

Scanning case folders: 100%|██████████| 501/501 [00:00<00:00, 651.10it/s]

Scanned 501 case folders.
Case-ID mismatches: 6
No T1c/T1gad folder found at all: 0

                 case_id                                           case_dir  \
0  UCSF-PDGM-0391_FU016d  /kaggle/input/datasets/usmansadiqcs/ucsf-pdgm-...   
1  UCSF-PDGM-0396_FU175d  /kaggle/input/datasets/usmansadiqcs/ucsf-pdgm-...   
2  UCSF-PDGM-0409_FU001d  /kaggle/input/datasets/usmansadiqcs/ucsf-pdgm-...   
3  UCSF-PDGM-0429_FU003d  /kaggle/input/datasets/usmansadiqcs/ucsf-pdgm-...   
4  UCSF-PDGM-0431_FU001d  /kaggle/input/datasets/usmansadiqcs/ucsf-pdgm-...   
5  UCSF-PDGM-0433_FU007d  /kaggle/input/datasets/usmansadiqcs/ucsf-pdgm-...   

                    matched_file  \
0  UCSF-PDGM-0289_T1gad_bias.nii   
1  UCSF-PDGM-0175_T1gad_bias.nii   
2  UCSF-PDGM-0181_T1gad_bias.nii   
3  UCSF-PDGM-0138_T1gad_bias.nii   
4  UCSF-PDGM-0278_T1gad_bias.nii   
5  UCSF-PDGM-0315_T1gad_bias.nii   

                                             warning  
0  WARNING: folder is for case UCSF-PDGM-0391_FU0... 

In [12]:
tar_files = glob.glob(f"{brats_root}/*.tar")
print(f"{len(tar_files)} tar files, {sum(os.path.getsize(t) for t in tar_files)/1e9:.1f} GB total")

3 tar files, 13.4 GB total


In [13]:
# does the "wrong" file's case number already exist as its own proper folder?
mismatch_df = pd.read_csv("/kaggle/working/ucsf_case_id_mismatches.csv")
mismatch_df["embedded_case_num"] = mismatch_df["matched_file"].str.extract(r"UCSF-PDGM-(\d+)")
for num in mismatch_df["embedded_case_num"]:
    exists = any(f"UCSF-PDGM-{num}_nifti" in c for c in case_dirs)
    print(f"UCSF-PDGM-{num}_nifti exists on its own: {exists}")

# is the mismatch just this one modality, or the whole case folder?
for _, row in mismatch_df.iterrows():
    print(f"\n--- {row['case_id']} ---")
    for mod in ["t1", "t1c", "t1gad", "t2", "flair", "segmentation"]:
        find_modality_file(row["case_dir"], [mod])

UCSF-PDGM-0289_nifti exists on its own: False
UCSF-PDGM-0175_nifti exists on its own: False
UCSF-PDGM-0181_nifti exists on its own: False
UCSF-PDGM-0138_nifti exists on its own: False
UCSF-PDGM-0278_nifti exists on its own: False
UCSF-PDGM-0315_nifti exists on its own: False

--- UCSF-PDGM-0391_FU016d ---

--- UCSF-PDGM-0396_FU175d ---

--- UCSF-PDGM-0409_FU001d ---

--- UCSF-PDGM-0429_FU003d ---

--- UCSF-PDGM-0431_FU001d ---

--- UCSF-PDGM-0433_FU007d ---


In [14]:
bad_ids = mismatch_df["case_id"].tolist()
case_dirs = [c for c in case_dirs if os.path.basename(c).split("_nifti")[0] not in bad_ids]
print(f"{len(case_dirs)} case folders remaining after excluding {len(bad_ids)} known mismatches.")

495 case folders remaining after excluding 6 known mismatches.


# PRE PROCESSING